# Clasificación de hongos con Regresión Logística

Este notebook muestra cómo construir un modelo de **clasificación binaria** usando **Regresión Logística** para la base de datos *Mushroom*.

La base de datos debe estar disponible localmente como archivo CSV. No se descarga nada desde internet.

El objetivo es clasificar cada hongo como:

- `e`: edible / comestible
- `p`: poisonous / venenoso

En esta base de datos, la variable objetivo se llama:

```python
class
```


## 1. Importar librerías

Se utilizan librerías básicas para lectura de datos, partición entrenamiento-validación, preprocesamiento, entrenamiento y evaluación.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_curve,
    roc_auc_score
)

import matplotlib.pyplot as plt


## 2. Cargar el archivo CSV

El archivo CSV debe estar en la misma carpeta donde se ejecuta este notebook.

Si el archivo tiene otro nombre, modifique únicamente la variable `CSV_FILE`.


In [ ]:
CSV_FILE = "mushrooms.csv"
TARGET_COLUMN = "class"

df = pd.read_csv(CSV_FILE)

df.head()


## 3. Exploración básica de los datos

Primero se revisa el tamaño de la base de datos, los nombres de las columnas y los tipos de variables.


In [ ]:
print("Filas y columnas:", df.shape)
print("\nColumnas:")
print(df.columns.tolist())
print("\nTipos de datos:")
print(df.dtypes)


## 4. Revisar la variable objetivo

La variable objetivo es `class`. Esta columna contiene las clases que se desean predecir.


In [ ]:
df[TARGET_COLUMN].value_counts()

In [ ]:
df[TARGET_COLUMN].value_counts(normalize=True)


## 5. Revisión de valores faltantes

En esta base algunos valores faltantes pueden aparecer representados con el símbolo `?`.

Para tratarlos de forma clara, se reemplazan por `NaN`.


In [ ]:
df = df.replace("?", np.nan)
df.isna().sum().sort_values(ascending=False).head(10)


## 6. Separar variables de entrada y variable objetivo

La matriz `X` contiene las características del hongo.

El vector `y` contiene la clase que se desea predecir.


In [ ]:
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

print("Tamaño de X:", X.shape)
print("Tamaño de y:", y.shape)


## 7. Codificación de la variable objetivo

Para calcular curva ROC y AUC conviene transformar la clase a valores numéricos.

Se usará la siguiente convención:

- `e` → 0
- `p` → 1

De esta forma, la clase positiva será el hongo venenoso.


In [ ]:
y_numeric = y.map({"e": 0, "p": 1})

if y_numeric.isna().any():
    raise ValueError("La variable objetivo contiene valores distintos de 'e' y 'p'. Revise la columna class.")

y_numeric.value_counts()


## 8. Partición en entrenamiento y validación

Se separan los datos en dos conjuntos:

- **Entrenamiento:** usado para ajustar el modelo.
- **Validación:** usado para evaluar el modelo con datos no vistos durante el entrenamiento.

Se usa `stratify=y_numeric` para mantener una proporción similar de clases en ambos conjuntos.


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y_numeric,
    test_size=0.2,
    random_state=42,
    stratify=y_numeric
)

print("Entrenamiento:", X_train.shape)
print("Validación:", X_val.shape)


## 9. Preprocesamiento de variables categóricas

La Regresión Logística trabaja con valores numéricos.

Como las variables de esta base son categóricas, se utiliza **One-Hot Encoding**. Este método convierte cada categoría en una columna binaria.

Por ejemplo, una variable como `color = rojo, verde, azul` se convierte en tres columnas indicadoras.


In [ ]:
categorical_features = X.columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)


## 10. Modelo de Regresión Logística

La Regresión Logística estima la probabilidad de pertenecer a una clase.

Para clasificación binaria, el modelo puede interpretarse como:

$$
P(y=1 \mid x) = \frac{1}{1 + e^{-z}}
$$

con:

$$
z = w_0 + w_1x_1 + w_2x_2 + \cdots + w_nx_n
$$

En este problema:

- `y = 0` representa hongo comestible.
- `y = 1` representa hongo venenoso.

Por tanto, el modelo estima la probabilidad de que un hongo sea venenoso.


In [ ]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                solver="lbfgs"
            )
        )
    ]
)

model


## 11. Entrenar el modelo

El entrenamiento consiste en ajustar los parámetros del modelo usando los datos de entrenamiento.


In [ ]:
model.fit(X_train, y_train)


## 12. Realizar predicciones

Después del entrenamiento, se predicen las clases del conjunto de validación.


In [ ]:
y_pred = model.predict(X_val)
y_proba = model.predict_proba(X_val)[:, 1]

pd.DataFrame({
    "y_real": y_val.values,
    "y_predicha": y_pred,
    "probabilidad_venenoso": y_proba
}).head()


## 13. Evaluación del modelo

Se calculan varias métricas de clasificación.

- **Accuracy:** proporción total de aciertos.
- **Precision:** de los hongos predichos como venenosos, cuántos realmente eran venenosos.
- **Recall:** de los hongos venenosos reales, cuántos fueron detectados.
- **F1-score:** promedio armónico entre precision y recall.

En problemas de seguridad, como detectar hongos venenosos, el **recall de la clase venenosa** es especialmente importante.


In [ ]:
accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

In [ ]:
print(classification_report(
    y_val,
    y_pred,
    target_names=["comestible", "venenoso"]
))


## 14. Matriz de confusión

La matriz de confusión permite observar los aciertos y errores del modelo.

Para esta convención:

- Clase 0: comestible
- Clase 1: venenoso

La situación más delicada sería clasificar un hongo venenoso como comestible.


In [ ]:
cm = confusion_matrix(y_val, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["comestible", "venenoso"]
)

disp.plot(values_format="d")
plt.title("Matriz de confusión - Regresión Logística")
plt.show()


## 15. Interpretación básica de coeficientes

Después del One-Hot Encoding, cada categoría se convierte en una variable binaria.

Los coeficientes positivos aumentan la probabilidad de clase `1`, es decir, venenoso.

Los coeficientes negativos disminuyen la probabilidad de clase `1`, es decir, se asocian más con la clase comestible.


In [ ]:
encoder = model.named_steps["preprocessor"].named_transformers_["categorical"]
feature_names = encoder.get_feature_names_out(categorical_features)

coefficients = model.named_steps["classifier"].coef_[0]

coef_df = pd.DataFrame({
    "caracteristica": feature_names,
    "coeficiente": coefficients
})

coef_df.sort_values("coeficiente", ascending=False).head(15)

In [ ]:
coef_df.sort_values("coeficiente", ascending=True).head(15)